Подготавливаем среду для работы:

In [1]:
!python --version

Python 3.11.13


In [1]:
!pip install tensorflow==2.15.0 keras==2.15.0

In [2]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [3]:
import tensorflow as tf
import keras
print('TF:', tf.__version__)
print('Keras:', keras.__version__)
print('Legacy Keras:', os.environ.get('TF_USE_LEGACY_KERAS'))

TF: 2.15.0
Keras: 2.15.0
Legacy Keras: 1


In [4]:
!pip install --pre deepchem

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 25.0 MB/s eta 0:00:00


Заглушаем предупреждения:

In [6]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [7]:
import deepchem as dc
from deepchem.models.graph_models import GraphConvModel

def solubility(split_method):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer='GraphConv', splitter=split_method
    )
    train, valid, test = datasets
    model = GraphConvModel(n_tasks=1, mode='regression', dropout=0.2)
    model.fit(train, nb_epoch=100)
    metric = dc.metrics.Metric(dc.metrics.mean_squared_error)
    train_score = model.evaluate(train, [metric], transformers)
    test_score = model.evaluate(test, [metric], transformers)
    print(f"Split: {split_method}")
    print(f"  Train RMSE: {train_score['mean_squared_error']:.4f}")
    print(f"  Test RMSE:  {test_score['mean_squared_error']:.4f}")
    return train_score, test_score

print("=" * 50)
solubility('random')
print("=" * 50)
solubility('scaffold')

Split: random
  Train RMSE: 0.7566
  Test RMSE:  1.0906
Split: scaffold
  Train RMSE: 0.4531
  Test RMSE:  1.5365


({'mean_squared_error': 0.45306254059247014},
 {'mean_squared_error': 1.5364863093571672})